# ThreatPrior AI: Dual-Stage AI Vulnerability Prioritization
### Explainable Threat Intelligence Synthesis, Strands Decider 2B (System-1), Amazon Bedrock (System-2), and Evidence-Based Risk Policy Validation

**Submission for AWS Builder Center / AWS AI Hackathon**  
**Repository**: [github.com/v-abhiramreddy/ThreatPrior-AI](https://github.com/v-abhiramreddy/ThreatPrior-AI)  
**Author**: Abhiram Reddy  
**Date**: October 2026  

---

## 1. Project Overview and Problem Statement

Modern enterprise Security Operations Centers (SOCs) and vulnerability management teams face an acute **vulnerability alert fatigue crisis**:

1. **CVSS Base Score Alert Fatigue**:
   - Over **40% of all published CVEs** in the NIST National Vulnerability Database (NVD) are assigned "High" or "Critical" ratings under CVSS v3.1 base metrics.
   - Base CVSS measures *theoretical technical severity in a vacuum* (e.g. theoretical impact if attack prerequisites are satisfied). It does not factor in whether exploit code is actively traded, weaponized, or utilized in real-world campaigns.
   - Remediating thousands of CVSS 9.0+ vulnerabilities within standard 7- to 14-day enterprise SLAs is mathematically and operationally impossible.

2. **The Pitfalls of Naive Single-Prompt LLM Evaluation**:
   - When engineering teams attempt to automate triage by feeding raw CVE descriptions into general-purpose LLMs, four dangerous failure modes emerge:
     - **Hallucinated Exploitation**: Claiming a vulnerability is actively exploited in the wild without verified intelligence.
     - **Prompt Injection Vulnerabilities**: Adversaries and security researchers regularly embed prompt injection payloads inside public CVE summaries and bug tracker descriptions.
     - **Uncalibrated Confidence**: Generative LLMs routinely declare 99% confidence ratings with zero mathematical grounding.
     - **Fabricated Remediation**: LLMs synthesize nonexistent target patch versions or propose destructive command-line scripts.

### The ThreatPrior AI Solution

ThreatPrior AI replaces static single-metric triage and brittle single-prompt LLM evaluation with a **truthful four-tier architecture**:
1. **Multi-Source Evidence Aggregation**: Concurrent ingestion of NIST NVD (technical base metrics), CISA KEV (confirmed active in-the-wild exploitation under BOD 22-01), and FIRST.org EPSS (empirical 30-day forecast estimating the probability that a vulnerability will be exploited in the wild).
2. **Stage 1 (System-1 Fast Decision Engine)**: Strands Decider 2B evaluates normalized evidence to output discrete priority choices (`Critical`, `High`, `Medium`, `Low`), probability distributions, uncalibrated confidence ratings, and emergency mitigation requirements.
3. **Stage 2 (System-2 Deep Reasoning Engine)**: Amazon Bedrock synthesizes grounded rationales and operational action playbooks under strict XML tag boundaries (`<untrusted_evidence_data>`).
4. **Evidence-Based Risk Policy Validation (`v2026.1-policy`)**: An independent deterministic policy auditor that verifies model decisions against enterprise rules without hallucination risk.

## 2. ThreatPrior AI Architecture and Two-Stage Decision Flow

ThreatPrior AI implements a clean separation between **fast, bounded System-1 classification** and **deep, grounded System-2 explanatory reasoning**:

```
[ Security Operator / API Request ]
                 │
                 ▼
     FastAPI Backend Core
                 │
    ┌────────────┴───────────────────────────┐
    ▼                                        ▼
[ NIST NVD REST v2 ]      [ CISA KEV JSON ]      [ FIRST.org EPSS v3 ]
    └────────────┬───────────────────────────┘
                 │ Normalized Telemetry & Source Provenance
                 ▼
    ┌────────────────────────────────────────────────────────┐
    │  Stage 1: Strands Decider 2B (System-1 Fast Decision)  │
    │  • Discrete Choice Classification (Critical/High/Med)  │
    │  • Multi-Hypothesis Probability Distribution           │
    │  • Emergency Out-of-Band Mitigation Requirement (Noul) │
    └────────────────────┬───────────────────────────────────┘
                         │ Stage 1 Decision State
                         ▼
    ┌────────────────────────────────────────────────────────┐
    │  Stage 2: Amazon Bedrock (System-2 Deep Reasoning)     │
    │  • Strict XML Data Isolation (<untrusted_evidence>)    │
    │  • Grounded Rationale & Operational Recommendations    │
    │  • Automated Version & Command Anti-Hallucination Audit│
    └────────────────────┬───────────────────────────────────┘
                         │
                         ▼
    ┌────────────────────────────────────────────────────────┐
    │  Evidence-Based Risk Validation Layer (v2026.1-policy) │
    │  • Deterministic Organizational Policy Compliance      │
    │  • Policy Alignment vs. Disagreement Flagging          │
    │  • Preservation of Upstream Failure Integrity          │
    └────────────────────┬───────────────────────────────────┘
                         │
                         ▼
             Structured Triage Decision
```

### Architectural Core Principles:
- **Separation of Model Provenance**: System-1 (`strands-decider-2b`) and System-2 (`au.anthropic.claude-sonnet-4-6` or `amazon.nova-lite-v1:0`) are tracked distinctly. The system never constructs deceptive hybrid labels.
- **Truthful Degradation**: If Bedrock fails or credentials expire, the engine cleanly degrades to `decider-only` mode; if both AI stages are bypassed, it falls back to a deterministic heuristic `baseline` without claiming AI execution.
- **Input Neutralization**: Untrusted telemetry is encapsulated in `<untrusted_evidence_data>` tags to neutralize prompt injection attacks.

## 3. Prerequisites and Environment Configuration

### Dependencies & Runtime Environment:
- **Python Runtime**: Python 3.10 to 3.12 (Python 3.12.10 verified in this project).
- **Core Dependencies** (from `backend/requirements.txt`):
  - `fastapi>=0.115.0`, `uvicorn[standard]>=0.32.0`, `httpx>=0.28.0`
  - `pydantic>=2.10.0`, `pydantic-settings>=2.6.0`, `python-dotenv>=1.0.1`
  - `boto3>=1.34.0`, `botocore>=1.34.0`
  - `strands-decider>=0.1.0`
- **Development & Testing Dependencies** (from `backend/requirements-dev.txt`):
  - `pytest>=8.3.0`, `pytest-asyncio>=0.24.0`, `respx>=0.22.0`

### Repository Path & Environment Resolution:
This notebook imports the real backend service modules from `backend/app/`. Running Jupyter from the repository root mounts the backend package onto `sys.path`.

### AWS Region and Model Configuration Resolution:
ThreatPrior AI supports multiple AWS deployment regions and models via environment settings:
- **Default Application Fallback** (in `backend/app/config.py` and `.env.example`):
  - `AWS_REGION=us-east-1`
  - `BEDROCK_MODEL_ID=amazon.nova-lite-v1:0` (or US regional profile `us.anthropic.claude-sonnet-4-6`)
- **Active Deployment Configuration** (in `backend/.env`):
  - `AWS_REGION=ap-southeast-2` (Sydney)
  - `BEDROCK_MODEL_ID=au.anthropic.claude-sonnet-4-6` (Anthropic Claude 3.7 / 3.5 Sonnet regional inference profile for Sydney)

**Important Prerequisite**: Whichever AWS region and model identifier are configured, the model must be available and access must be requested/enabled in the AWS Bedrock Console for that region.

### Security & Credential Hygiene:
- **Zero Hardcoded Secrets**: No AWS secret keys, session tokens, or API credentials are ever written into this notebook or its outputs.
- **AWS Authentication**: Authentication is resolved via standard AWS SDK provider chains: AWS environment variables (`AWS_ACCESS_KEY_ID`, `AWS_SECRET_ACCESS_KEY`), AWS profiles (`~/.aws/credentials`), or IAM instance roles.

In [1]:
# Section 3: Environment Configuration Discovery & Production Module Imports
import sys
import os
from pathlib import Path

# Anchor path resolution to the repository and backend directory
repo_root = Path.cwd()
backend_dir = repo_root / "backend"
if str(backend_dir) not in sys.path:
    sys.path.insert(0, str(backend_dir))

# Import actual production modules from ThreatPrior AI
from app.config import settings
from app.models.enums import OverallStatus, SeverityLevel, SourceStatus
from app.models.schemas import VulnerabilityResponse
from app.models.triage import (
    DecisionEngineType,
    ModelInvocationStatus,
    RiskValidationStatus,
    TriageDecision,
    TriagePriority,
    Stage2BedrockOutput,
)
from app.services.aggregator import aggregator, generate_operational_summary
from app.services.cisa_client import cisa_client
from app.services.epss_client import epss_client
from app.services.nvd_client import nvd_client
from app.services.risk_validator import risk_validator
from app.services.strands_client import strands_client, DeciderEvaluation, StrandsDeciderClient
from app.services.decision_engine import (
    decision_engine,
    DecisionEngine,
    STAGE2_SYSTEM_PROMPT,
    classify_bedrock_error,
    diagnose_bedrock_error,
)

print("=" * 70)
print(f"Service Name           : {settings.app_name}")
print(f"Service Version        : {settings.app_version}")
print(f"Configured AWS Region  : {settings.aws_region}")
print(f"Configured Bedrock ID  : {settings.bedrock_model_id}")
print(f"Bedrock Max Tokens     : {settings.bedrock_max_tokens}")
print(f"Strands Decider Model  : {settings.strands_decider_model_id} (mode={settings.strands_decider_mode})")
print(f"Strands Decider URL    : {settings.strands_decider_url}")
print(f"Baseline Fallback OK   : {settings.allow_baseline_fallback}")
print("=" * 70)

Service Name           : ThreatPrior AI Vulnerability Intelligence Service
Service Version        : 0.3.0
Configured AWS Region  : ap-southeast-2
Configured Bedrock ID  : au.anthropic.claude-sonnet-4-6
Bedrock Max Tokens     : 2048
Strands Decider Model  : strands-decider-2b (mode=remote)
Strands Decider URL    : https://api.decider.cloud
Baseline Fallback OK   : True

## 4. NVD, CISA KEV, and EPSS Evidence Collection

ThreatPrior AI queries three authoritative threat intelligence sources concurrently via `VulnerabilityAggregator`:

| Threat Intelligence Stream | Primary Role | Critical Distinction & Safety Invariant |
|:---|:---|:---|
| **NIST NVD REST v2.0** | Authoritative technical vulnerability description, CVSS v3.1/v2 base scores, vector strings, CWEs, and affected CPE configurations. | **Theoretical Severity**: Measures technical impact in a vacuum. If NVD is rate-limited or fails, status is preserved as `FAILED` or `RATE_LIMITED`; metrics are never fabricated. |
| **CISA KEV Catalog** | Confirmed in-the-wild active exploitation under Binding Operational Directive (BOD) 22-01. | **Active Exploitation Confirmation**: Absence from KEV (`NOT_LISTED`) indicates no confirmed federal campaign under BOD 22-01, **NEVER proof of safety**. |
| **FIRST.org EPSS v3** | Empirical forecast estimating the probability that a vulnerability will be exploited in the wild over the next 30 days. | **Predictive Likelihood**: EPSS is a 30-day statistical forecast; it is **NEVER proof that exploitation has occurred**. |

These signals provide **complementary, distinct evidence** rather than interchangeable measurements.

The following cell queries live threat intelligence for **CVE-2021-44228** (Apache Log4Shell):

In [2]:
# Section 4: Multi-Source Threat Intelligence Retrieval (CVE-2021-44228)
import asyncio
import json

# Ensure bundled CISA KEV seed snapshot is available for fallback resilience
cisa_client.load_seed_catalog()

cve_id = "CVE-2021-44228"
print(f"Fetching fresh threat intelligence across NVD, CISA KEV, and EPSS for {cve_id}...")

# Execute concurrent multi-source aggregation via production aggregator
evidence_resp: VulnerabilityResponse = asyncio.run(aggregator.aggregate_cve(cve_id))

print(f"\n[+] Aggregation Status : {evidence_resp.status.value.upper()}")
print(f"[+] Retrieval Timestamp: {evidence_resp.retrieved_at.isoformat()}")

prov = evidence_resp.provenance
norm = evidence_resp.normalized

print("\n" + "-" * 50)
print("1. NIST NVD REST API v2.0 Telemetry (Theoretical Severity):")
print(f"   - Source Status     : {prov.nvd.status.value}")
print(f"   - Published Date    : {prov.nvd.published_date}")
print(f"   - CVSS Base Score   : {norm.cvss_score if norm else 'N/A'} ({norm.primary_severity.value if norm else 'N/A'})")
print(f"   - CVSS Vector       : {norm.cvss_vector if norm else 'N/A'}")
print(f"   - Weaknesses (CWE)  : {', '.join(prov.nvd.weaknesses)}")
print(f"   - Affected CPE Count: {norm.affected_cpe_count if norm else 0}")

print("\n2. CISA KEV Catalog Telemetry (Active Exploitation Confirmation):")
print(f"   - Source Status     : {prov.cisa_kev.status.value}")
print(f"   - Is Listed in KEV  : {prov.cisa_kev.is_listed}")
print(f"   - Date Added to KEV : {prov.cisa_kev.date_added}")
print(f"   - BOD Due Date      : {prov.cisa_kev.due_date}")
print(f"   - Ransomware Link   : {prov.cisa_kev.known_ransomware_campaign_use}")
print(f"   - Authoritative Act.: {prov.cisa_kev.required_action[:120]}...")

print("\n3. FIRST.org EPSS v3 Telemetry (30-Day Exploitation Probability Forecast):")
print(f"   - Source Status     : {prov.epss.status.value}")
if prov.epss.probability is not None:
    print(f"   - 30-Day In-Wild Exploitation Probability Estimate: {prov.epss.probability * 100:.2f}% (Percentile: {prov.epss.percentile * 100:.2f}%)")
print(f"   - Score Date        : {prov.epss.score_date}")
print(f"   - Interpretive Note : {prov.epss.interpretive_note}")

print("\n" + "-" * 50)
print("Plain-English Operational Risk Summary:")
print(norm.summary if norm else "N/A")

Fetching fresh threat intelligence across NVD, CISA KEV, and EPSS for CVE-2021-44228...

[+] Aggregation Status : OK
[+] Retrieval Timestamp: 2026-10-24T06:02:29.552274+00:00

--------------------------------------------------
1. NIST NVD REST API v2.0 Telemetry (Theoretical Severity):
   - Source Status     : AVAILABLE
   - Published Date    : 2021-12-10T10:15:09.143
   - CVSS Base Score   : 10.0 (CRITICAL)
   - CVSS Vector       : CVSS:3.1/AV:N/AC:L/PR:N/UI:N/S:C/C:H/I:H/A:H
   - Weaknesses (CWE)  : CWE-20, CWE-400, CWE-502, CWE-917
   - Affected CPE Count: 381

2. CISA KEV Catalog Telemetry (Active Exploitation Confirmation):
   - Source Status     : LISTED
   - Is Listed in KEV  : True
   - Date Added to KEV : 2021-12-10
   - BOD Due Date      : 2021-12-24
   - Ransomware Link   : Known
   - Authoritative Act.: For all affected software assets for which updates exist, the only acceptable remediation actions are: 1) Apply updates;...

3. FIRST.org EPSS v3 Telemetry (30-Day Exploitat

## 5. Actual Strands Decider Execution

In ThreatPrior AI, **Stage 1 triage prioritization is executed by Strands Decider 2B**, a specialized 2-billion parameter decision model:

### System-1 Decision Mechanics:
1. **Discrete Multi-Choice Classification**: Evaluates normalized threat intelligence against strict criteria for `Critical`, `High`, `Medium`, and `Low` priority.
2. **Probability Distribution**: Returns raw model softmax probabilities across all 4 hypotheses.
3. **Emergency Mitigation (Noul Question)**: Evaluates a dedicated binary operational question (`requires_emergency_mitigation`) with a floating-point score between 0.0 and 1.0. A score $\ge 0.50$ triggers immediate out-of-band mitigation flags.
4. **Normalized Distribution Concentration (Confidence)**: Computed per the official Strands Decider formula:
   $$\text{Confidence} = \frac{N \cdot p_{\max} - 1}{N - 1}$$
   where $N=4$ and $p_{\max}$ is the maximum probability across the 4 priority choices.
5. **Transparency Invariant**: ThreatPrior AI explicitly sets `confidence_is_calibrated: False`, transparently acknowledging that distribution concentration is an uncalibrated peakedness score.

The following cell formats the evidence state and executes live System-1 inference against the configured remote Strands Decider endpoint (`https://api.decider.cloud`):

In [3]:
# Section 5: Stage 1 -- Strands Decider 2B Fast System-1 Inference (Live Execution)
state_prompt = strands_client.format_evidence_state(evidence_resp)
questions = strands_client.build_systemone_questions()

print("--- Formatted Strands Decider Evidence State (Input Context) ---")
print(state_prompt)

print("\n--- Official System-1 Choice and Noul Questions Schema ---")
print(json.dumps(questions, indent=2))

print("\n" + "=" * 50)
print("Invoking Strands Decider 2B System-1 inference...")
decider_eval: DeciderEvaluation = asyncio.run(strands_client.evaluate(evidence_resp))

print(f"\n[+] Strands Decider Model Checkpoint : {decider_eval.model_id}")
print(f"[+] Assigned Triage Priority         : {decider_eval.priority.value.upper()}")
print(f"[+] Model Confidence Score           : {decider_eval.confidence * 100:.2f}% (distribution concentration)")
print(f"[+] Confidence Calibrated Status     : {decider_eval.confidence_is_calibrated} (uncalibrated peakedness)")
print(f"[+] Emergency Mitigation Noul Score  : {decider_eval.emergency_mitigation_noul:.4f}")
print(f"[+] Emergency Mitigation Required    : {decider_eval.requires_emergency_mitigation}")

print("\nProbability Distribution Across Triage Classes:")
for prio, p in decider_eval.priority_probabilities.items():
    bar_len = int(p * 30)
    bar = "#" * bar_len + "-" * (30 - bar_len)
    print(f"  {prio:<8}: [{bar}] {p * 100:5.2f}%")

--- Formatted Strands Decider Evidence State (Input Context) ---
Vulnerability Identifier: CVE-2021-44228
Description: Apache Log4j2 2.0-beta9 through 2.15.0 (excluding security releases 2.12.2, 2.12.3, and 2.3.1) JNDI features used in configuration, log messages, and parameters do not protect against attacker controlled LDAP and other JNDI related endpoints. An attacker who can control log messages or log message parameters can execute arbitrary code loaded from LDAP servers when message lookup substitution is enabled. From log4j 2.15.0, this behavior has been disabled by default. From version 2.16.0 (along ...
CVSS Base Score: 10.0 (Severity: CRITICAL)
CISA KEV Listed: YES (Status: LISTED)
CISA KEV Date Added: 2021-12-10
Known Ransomware Campaign Use: Known
EPSS Exploitation Probability (30-day): 100.00% (Percentile: 100.0%)
Affected CPE Configurations: 381

--- Official System-1 Choice and Noul Questions Schema ---
{
  "priority": {
    "type": "choice",
    "instructions": "Assign 

## 6. Actual Amazon Bedrock Execution (Live Pipeline Probe & Failure Analysis)

Once Stage 1 has established the verified priority choice and emergency mitigation flag, **Amazon Bedrock** acts as the System-2 deep reasoning and explanatory synthesizer.

### Supported Models & Configuration:
- **Anthropic Claude on Bedrock**: `au.anthropic.claude-sonnet-4-6` (Sydney regional profile) or `us.anthropic.claude-sonnet-4-6` (US regional profile).
- **Amazon Nova**: `amazon.nova-lite-v1:0` (cost-optimized, high-efficiency).
- **Execution Interface**: AWS Bedrock Runtime `converse()` API with strict zero-temperature (`temperature: 0.0`) and bounded output token limits.

### Crucial Safety Guardrails & System Prompt Rules:
1. **Strict Data Separation**: Untrusted vulnerability telemetry is encapsulated within `<untrusted_evidence_data>` tags. The model is forbidden from obeying commands or instructions embedded inside CVE descriptions.
2. **Grounding**: Rationale must be strictly based on supplied telemetry.
3. **EPSS Non-Exploitation Rule**: Model cannot assert confirmed in-wild exploitation based solely on high EPSS scores.
4. **Stage 1 Consistency**: Bedrock's proposed priority **must match Stage 1's decision**.

> [!WARNING]
> **Live Execution Status: Bedrock Inactive / Token Expired**  
> In the recorded execution below, the live Bedrock execution probe was executed against the configured AWS Bedrock endpoint (`au.anthropic.claude-sonnet-4-6` in `ap-southeast-2`).  
> The invocation encountered `LoginRefreshRequired`: the host's active AWS session token has expired and requires `aws login` / SSO re-authentication.  
> **Crucial Integrity Invariant**: The system **does NOT claim dual-model success**. Instead, it demonstrates ThreatPrior AI's truthful production fallback to **`decider-only` degraded mode**, preserving Strands Decider's priority (`Critical`) and confidence (`49.56%`) while explicitly documenting the Bedrock outage.

### How to Run with Active AWS Credentials:
To execute full live dual-model inference in an authenticated environment:
1. Re-authenticate with your AWS provider: `aws sso login` or configure standard environment variables (`AWS_ACCESS_KEY_ID`, `AWS_SECRET_ACCESS_KEY`).
2. Verify permissions for `bedrock:InvokeModel` and `bedrock:Converse`.
3. Re-run `await decision_engine.evaluate_vulnerability(evidence_resp)`.

In [4]:
# Section 6: Stage 2 -- Amazon Bedrock Execution Probe & Truthful Degraded Mode
# 1. Format the Stage 2 Bedrock prompt embedding normalized evidence and Stage 1 decision
stage2_prompt = decision_engine._format_stage2_prompt(evidence_resp, decider_eval)

print("--- Amazon Bedrock Stage 2 Prompt Structure (Excerpt) ---")
print(stage2_prompt[:1100] + "\n... [evidence payload formatted inside <untrusted_evidence_data> tags]")

print("\n--- Amazon Bedrock System Prompt Excerpt ---")
print(STAGE2_SYSTEM_PROMPT[:450] + "\n... [enforcing grounding, JSON schema, and strict length bounds]")

print("\n" + "=" * 50)
print("Executing decision_engine.evaluate_vulnerability() live pipeline probe...")

# 2. Execute decision_engine pipeline with live AWS Bedrock probe
live_triage = asyncio.run(decision_engine.evaluate_vulnerability(evidence_resp))

print(f"\n[+] Execution Engine Mode    : {live_triage.engine.value}")
print(f"[+] Final Triage Priority    : {live_triage.priority.value}")
print(f"[+] Strands Decider Status   : {live_triage.decider_status.value} (Model: {live_triage.decider_model_id})")
print(f"[+] Amazon Bedrock Status    : {live_triage.bedrock_status.value} (Model: {live_triage.bedrock_model_id})")
print(f"[+] Fallback / Degraded Info : {live_triage.fallback_reason}")
print(f"[+] Synthesized Rationale    : {live_triage.rationale}")
print(f"[+] Recommended Action       : {live_triage.recommended_action}")

--- Amazon Bedrock Stage 2 Prompt Structure (Excerpt) ---
Stage 1 Strands Decider 2B System-1 fast evaluation has concluded with the following verified decision:
- Assigned Priority: Critical
- Model Checkpoint: strands-decider-2b
- Model-Derived Confidence: 49.56%
- Probability Distribution: {"Critical": 0.621738, "High": 0.348365, "Medium": 0.021998, "Low": 0.007899}
- Emergency Operational Mitigation Required (Noul): Yes (score: 0.702)

Untrusted vulnerability telemetry evidence:
<untrusted_evidence_data>
{
  "cve_id": "CVE-2021-44228",
  "overall_status": "OK",
  "evidence_retrieved_at": "2026-10-24T06:02:29.552274+00:00",
  "nvd_telemetry": {
    "source": "NIST National Vulnerability Database API v2.0",
    "status": "AVAILABLE",
    "description": "Apache Log4j2 2.0-beta9 through 2.15.0 (excluding security releases 2.12.2, 2.12.3, and 2.3.1) JNDI features used in configuration, log messages, and parameters do not protect against attacker controlled LDAP and other JNDI related en

## 7. Evidence-Based Validation and Operational Prioritization

AI models provide valuable decision support, but mission-critical security operations require **deterministic organizational policy compliance**.

ThreatPrior AI implements an independent, deterministic **Evidence-Based Risk Validation Engine** (`v2026.1-policy`):
- **Independent Audit**: Evaluates normalized CVSS, CISA KEV, and EPSS telemetry against documented organizational rules without overriding the model's priority.
- **Alignment vs. Disagreement**: Classifies the decision as `ALIGNED` (model matches policy) or `DISAGREEMENT` (model diverges from policy).
- **Preservation of Upstream Integrity**: If CISA KEV query failed, policy returns `INSUFFICIENT_EVIDENCE`—it **never assumes an unverified vulnerability is safe**.

### Documented Policy Ruleset:
- **Policy Rule 1 (Critical Severity)**: Confirmed CISA KEV active exploitation + (CVSS $\ge 8.5$ or EPSS $\ge 0.50$), OR extreme unmitigated CVSS $\ge 9.5$.
- **Policy Rule 2 (High Severity)**: Any confirmed CISA KEV listing, OR elevated CVSS $\ge 7.0$, OR high EPSS $\ge 0.20$.
- **Policy Rule 3 (Medium Severity)**: Moderate CVSS ($4.0 \le \text{CVSS} < 7.0$) or moderate EPSS ($\ge 0.05$) with no KEV listing.
- **Policy Rule 4 (Low Severity)**: Low CVSS ($< 4.0$), low EPSS ($< 0.05$), and confirmed KEV absence.

The following cell audits the Log4Shell triage decision against organizational policy:

In [5]:
# Section 7: Evidence-Based Risk Validation Layer (v2026.1-policy)
# Validate the model's triage decision against the deterministic organizational policy rules
policy_result = risk_validator.validate(evidence_resp, decider_eval.priority)

print(f"[+] Policy Ruleset Version   : {policy_result.validation_rules_version}")
print(f"[+] Model Assigned Priority  : {policy_result.model_priority.value}")
print(f"[+] Evidence Policy Priority : {policy_result.evidence_priority.value if policy_result.evidence_priority else 'None'}")
print(f"[+] Risk Validation Status   : {policy_result.risk_validation_status.value.upper()}")
print(f"[+] Evaluated Policy Rule    : {policy_result.policy_rule}")

print("\nPolicy Audit Explanation:")
print(policy_result.validation_explanation)

print("\nFactual Evidence Findings Audited by Policy:")
for factor in policy_result.risk_factors:
    print(f"  * {factor}")

[+] Policy Ruleset Version   : 2026.1-policy
[+] Model Assigned Priority  : Critical
[+] Evidence Policy Priority : Critical
[+] Risk Validation Status   : ALIGNED
[+] Evaluated Policy Rule    : Policy Rule 1 (Critical Severity)

Policy Audit Explanation:
Policy Aligned: Model-assigned 'Critical' priority is fully consistent with deterministic evidence policy rules (v2026.1-policy). Grounded trigger: Confirmed CISA KEV active exploitation combined with critical impact (CVSS >= 8.5 or high EPSS).

Factual Evidence Findings Audited by Policy:
  * NVD CVSS Base Score: 10.0 (CRITICAL) [Published: 2021-12-10T10:15:09.143].
  * CISA KEV: Listed as confirmed active threat (Added: 2021-12-10 · Ransomware: Known) [Catalog Version: 2026.10.23].
  * FIRST.org EPSS: 100.00% 30-day exploit likelihood, Top 0.0% (Observed on 2026-10-23 · Project heuristic threshold: >=20%).

## 8. Fallback and Error-Handling Demonstration

Reliable security engineering requires testing edge cases where telemetry is corrupted, sources fail, or AI models propose non-compliant priorities.

ThreatPrior AI implements 4 distinct engine operating modes:

| Engine Mode (`DecisionEngineType`) | Strands Decider 2B | Amazon Bedrock | Baseline Heuristic | Description |
|:---|:---:|:---:|:---:|:---|
| `dual-ai` | **SUCCESS** | **SUCCESS** | Skipped | Full production mode. Both models executed and verified. |
| `decider-only` | **SUCCESS** | **FAILED** | Skipped | Bedrock unavailable or session expired. System-1 priority and confidence preserved; rationale marked degraded. |
| `baseline` | Skipped / Failed | Skipped / Failed | **SUCCESS** | AI engines offline or bypassed. Deterministic heuristic baseline applied with transparent labeling. |
| `unavailable` | **FAILED** | **FAILED** | Disabled | Baseline fallback disabled; service raises explicit HTTP 503 error. |

The following cell demonstrates three critical safety edge cases:
1. **Edge Case A (Policy Disagreement)**: An unlisted CVE with extreme CVSS 9.8 base score assigned `High` by an AI model. Policy Rule 1 mandates `Critical`, surfacing an explicit `DISAGREEMENT` discrepancy summary.
2. **Edge Case B (Insufficient Evidence)**: CISA KEV query encountered a 503 error. The policy refuses to evaluate and returns `INSUFFICIENT_EVIDENCE`, preventing false safety assumptions.
3. **Deterministic Heuristic Baseline Fallback**: Verified baseline evaluation when AI engines are offline or bypassed.

In [6]:
# Section 8: Fallback, Edge Cases & Baseline Engine Verification
from app.models.schemas import CvssMetrics, NvdProvenance, CisaKevProvenance, EpssProvenance, Provenance, NormalizedVulnerability

print("--- Edge Case A: Policy Disagreement Detection ---")
# Model predicts High, but Policy Rule 1 mandates Critical for extreme CVSS (>= 9.5)
unlisted_critical_cve = VulnerabilityResponse(
    cve_id="CVE-2024-99999",
    status=OverallStatus.OK,
    retrieved_at=evidence_resp.retrieved_at,
    normalized=NormalizedVulnerability(
        title="Unlisted Critical Remote Code Execution",
        description="Remote code execution with CVSS 9.8 in core enterprise daemon.",
        summary="Remote code execution in core enterprise daemon.",
        primary_severity=SeverityLevel.CRITICAL,
        cvss_score=9.8,
        cvss_version="3.1",
        cvss_vector="CVSS:3.1/AV:N/AC:L/PR:N/UI:N/S:U/C:H/I:H/A:H",
        is_cisa_kev=False,
        epss_probability=0.10,
        epss_percentile=0.70,
        affected_cpe_count=1,
    ),
    provenance=Provenance(
        nvd=NvdProvenance(source="NVD", retrieved_at=evidence_resp.retrieved_at, status=SourceStatus.AVAILABLE, cvss_v31=CvssMetrics(base_score=9.8, severity=SeverityLevel.CRITICAL, vector_string="CVSS:3.1/AV:N/AC:L/PR:N/UI:N/S:U/C:H/I:H/A:H")),
        cisa_kev=CisaKevProvenance(source="CISA KEV", retrieved_at=evidence_resp.retrieved_at, status=SourceStatus.NOT_LISTED, is_listed=False, notes="Not listed in CISA KEV catalog."),
        epss=EpssProvenance(source="EPSS", retrieved_at=evidence_resp.retrieved_at, status=SourceStatus.AVAILABLE, probability=0.10, percentile=0.70),
    ),
)
discrepancy_result = risk_validator.validate(unlisted_critical_cve, TriagePriority.HIGH)
print(f"[+] Discrepancy Status       : {discrepancy_result.risk_validation_status.value.upper()}")
print(f"[+] Evaluated Policy Rule    : {discrepancy_result.policy_rule}")
print(f"[+] Discrepancy Summary      : {discrepancy_result.discrepancy_summary}")

print("\n--- Edge Case B: Insufficient Evidence Handling (No False Safety) ---")
# CISA KEV check failed - Policy refuses to assume safety (never treated as NOT_LISTED)
failed_kev_cve = VulnerabilityResponse(
    cve_id="CVE-2024-88888",
    status=OverallStatus.PARTIAL,
    retrieved_at=evidence_resp.retrieved_at,
    normalized=None,
    provenance=Provenance(
        nvd=NvdProvenance(source="NVD", retrieved_at=evidence_resp.retrieved_at, status=SourceStatus.NOT_FOUND),
        cisa_kev=CisaKevProvenance(source="CISA KEV", retrieved_at=evidence_resp.retrieved_at, status=SourceStatus.FAILED, is_listed=False, notes="CISA KEV query failed.", error_message="HTTP 503 Service Unavailable"),
        epss=EpssProvenance(source="EPSS", retrieved_at=evidence_resp.retrieved_at, status=SourceStatus.NOT_FOUND),
    ),
    errors=["CISA KEV: HTTP 503 Service Unavailable"],
)
insufficient_result = risk_validator.validate(failed_kev_cve, TriagePriority.MEDIUM)
print(f"[+] Insufficient Status      : {insufficient_result.risk_validation_status.value.upper()}")
print(f"[+] Explanation              : {insufficient_result.validation_explanation}")

print("\n--- Deterministic Heuristic Baseline Fallback Engine ---")
# Verify baseline evaluation when AI engines are offline or unauthorized
baseline_decision = decision_engine.evaluate_baseline(
    evidence_resp,
    fallback_reason="AI inference bypassed for baseline verification."
)
print(f"[+] Baseline Engine Mode     : {baseline_decision.engine.value}")
print(f"[+] Baseline Priority        : {baseline_decision.priority.value}")
print(f"[+] Baseline Confidence      : {baseline_decision.confidence:.2f} (Calibrated: {baseline_decision.confidence_is_calibrated})")
print(f"[+] Confidence Explanation   : {baseline_decision.confidence_explanation}")
print(f"[+] Fallback Reason          : {baseline_decision.fallback_reason}")
print(f"[+] Authoritative Directive  : {baseline_decision.authoritative_directive[:80]}...")

--- Edge Case A: Policy Disagreement Detection ---
[+] Discrepancy Status       : DISAGREEMENT
[+] Evaluated Policy Rule    : Policy Rule 1 (Critical Severity)
[+] Discrepancy Summary      : Policy Rule 1 (Critical Severity) mandates Critical priority for extreme CVSS (9.8 >= 9.5) regardless of CISA KEV listing status. Responsible Evidence: NVD CVSS is 9.8 (Critical) while CISA KEV status is NOT_LISTED, leading the AI model to recommend High whereas organizational policy enforces Critical.

--- Edge Case B: Insufficient Evidence Handling (No False Safety) ---
[+] Insufficient Status      : INSUFFICIENT_EVIDENCE
[+] Explanation              : Evidence-Based Validation Inconclusive: CISA KEV catalog lookup FAILED. Per validation policy, failed lookups cannot be treated as NOT_LISTED. CVSS base score is unavailable from NVD. Available evidence is insufficient to evaluate against organizational risk policy rules.

--- Deterministic Heuristic Baseline Fallback Engine ---
[+] Baseline Engine

## 9. Clearly Separated Synthetic Parser and Guardrail Test (Unit Test Fixture)

> [!IMPORTANT]
> **NOTICE: SYNTHETIC TEST FIXTURE (NOT A LIVE BEDROCK EXECUTION)**  
> The test below is an isolated unit test of ThreatPrior AI's production JSON parser (`_parse_and_validate_ai_json`) and automated remediation fact-checker (`_detect_unsupported_remediation_claims`).  
> It uses an offline JSON fixture to verify Pydantic schema validation, Stage 1 priority consistency enforcement, and anti-hallucination version tagging.  
> **This test does NOT represent a live Bedrock call or live dual-model execution.** All variables and outputs in this section are isolated (`synthetic_fixture_*`) and cannot overwrite or be confused with the live pipeline results in Section 6.

### Remediation Guidance Audit: The Case of Log4j `2.17.1`
In the synthetic fixture below, the model recommendation includes:  
`"Apply emergency patches per CISA BOD 22-01 directive and upgrade to version 2.17.1 immediately."`

- **Technical Context**: For Java 8+ runtimes, Apache Log4j `2.17.1` resolved CVE-2021-44228 alongside subsequent related issues (CVE-2021-45046, CVE-2021-45105, CVE-2021-44832). However, for systems running Java 7, Apache released `2.12.4`, and for Java 6, Apache released `2.3.2`. A universal recommendation of `2.17.1` is inappropriate for legacy enterprise branches.
- **ThreatPrior AI's Automated Guardrail**: Because the specific version `2.17.1` is not documented in the upstream NVD/CISA evidence text, ThreatPrior AI's fact-checker automatically tags it:
  `"Remediation target version '2.17.1' is model-synthesized and not documented in upstream NVD/CISA evidence; verify with official vendor advisories."`
- **Authoritative Directive Preservation**: ThreatPrior AI retains the official CISA KEV required action directive (`https://www.cisa.gov/uscert/ed-22-02-apache-log4j-recommended-mitigation-measures`) and directs operators to vendor release notes (`https://logging.apache.org/log4j/2.x/security.html`).

In [7]:
# Section 9: Synthetic Unit Test Fixture -- Response Parser & Guardrail Inspection
# PROMINENT NOTICE: This cell evaluates an OFFLINE SYNTHETIC FIXTURE to verify parser logic.
# It is completely separate from live pipeline execution and uses dedicated 'synthetic_' variables.

synthetic_fixture_bedrock_json = json.dumps({
    "priority": "Critical",
    "rationale": "Active in-the-wild exploitation confirmed via CISA KEV catalog combined with maximum CVSS 10.0 score.",
    "risk_factors": [
        "Active in-the-wild exploitation confirmed in CISA KEV catalog under BOD 22-01.",
        "CVSS base metric rated CRITICAL (10.0) with unauthenticated remote network attack vector.",
        "FIRST.org EPSS 30-day in-wild exploitation probability estimate is 100.00%."
    ],
    "recommended_action": "Apply emergency patches per CISA BOD 22-01 directive and upgrade to version 2.17.1 immediately.",
    "evidence_limitations": [
        "EPSS represents a 30-day empirical forecast, not historical incident telemetry."
    ]
})

print("Synthetic Test Fixture Input (Mock Stage 2 JSON):")
print(synthetic_fixture_bedrock_json)

# Parse and validate through ThreatPrior AI's production parser
synthetic_test_decision = decision_engine._parse_and_validate_ai_json(
    synthetic_fixture_bedrock_json,
    evidence_resp,
    decider_eval
)

print("\n" + "-" * 50)
print("[+] Test Fixture Mode           : UNIT TEST (SYNTHETIC PARSER FIXTURE)")
print(f"[+] Parsed Engine Output        : {synthetic_test_decision.engine.value}")
print(f"[+] Validated Priority          : {synthetic_test_decision.priority.value}")
print(f"[+] Decider Model Checkpoint    : {synthetic_test_decision.decider_model_id}")
print(f"[+] Bedrock Model ID            : {synthetic_test_decision.bedrock_model_id}")
print(f"[+] Recommendation Source       : {synthetic_test_decision.recommendation_source}")
print(f"[+] Recommendation Verified     : {synthetic_test_decision.recommendation_is_verified}")
print(f"[+] Authoritative Directive     : {synthetic_test_decision.authoritative_directive[:80]}...")
print(f"[+] Authoritative Directive URL : {synthetic_test_decision.authoritative_directive_url}")

print("\nGuardrail Disclosures Attached by Production Parser:")
for i, lim in enumerate(synthetic_test_decision.evidence_limitations, 1):
    print(f"  {i}. {lim}")

Synthetic Test Fixture Input (Mock Stage 2 JSON):
{"priority": "Critical", "rationale": "Active in-the-wild exploitation confirmed via CISA KEV catalog combined with maximum CVSS 10.0 score.", "risk_factors": ["Active in-the-wild exploitation confirmed in CISA KEV catalog under BOD 22-01.", "CVSS base metric rated CRITICAL (10.0) with unauthenticated remote network attack vector.", "FIRST.org EPSS 30-day in-wild exploitation probability estimate is 100.00%."], "recommended_action": "Apply emergency patches per CISA BOD 22-01 directive and upgrade to version 2.17.1 immediately.", "evidence_limitations": ["EPSS represents a 30-day empirical forecast, not historical incident telemetry."]}

--------------------------------------------------
[+] Test Fixture Mode           : UNIT TEST (SYNTHETIC PARSER FIXTURE)
[+] Parsed Engine Output        : dual-ai
[+] Validated Priority          : Critical
[+] Decider Model Checkpoint    : strands-decider-2b
[+] Bedrock Model ID            : au.anthrop

## 10. Results, Limitations, and Reproducibility Notes

### Q&A

**Q: How does ThreatPrior AI solve the alert fatigue crisis in vulnerability management?**  
**A:** Base CVSS metrics rate over 40% of CVEs as High or Critical because they measure theoretical impact rather than active threat activity. ThreatPrior AI fuses NIST NVD technical metrics with live CISA KEV in-the-wild exploitation catalogs and FIRST.org EPSS (estimating the probability that a vulnerability will be exploited in the wild over the next 30 days). It then applies Strands Decider 2B for fast System-1 classification and Amazon Bedrock for explainable System-2 operational reasoning.

**Q: Why use a two-stage architecture instead of a single LLM prompt?**  
**A:** Single-prompt LLM triage suffers from high latency, prompt injection risks from untrusted CVE descriptions, and ungrounded hallucinations. By decoupling fast, bounded choice classification (Strands Decider 2B) from grounded explanatory synthesis (Amazon Bedrock under `<untrusted_evidence_data>` tags), ThreatPrior AI achieves deterministic priority boundaries with auditability.

**Q: How does the system handle AI failures or credential expiration?**  
**A:** ThreatPrior AI never claims AI execution when an engine fails. When Bedrock session tokens expire or rate limits are reached, the system gracefully degrades to `decider-only` mode or deterministic heuristic `baseline`, maintaining uninterrupted SOC operations with transparent audit logs.

---

### Data Analysis Key Findings

- **Multi-Source Evidence Convergence**: For **CVE-2021-44228**, NIST NVD confirms a **10.0 CVSS base score**, CISA KEV confirms active in-the-wild exploitation under Binding Operational Directive 22-01, and FIRST.org EPSS reflects an empirical **100.00% 30-day in-wild exploitation probability estimate** (99.99th percentile).
- **System-1 Decision Distribution**: Strands Decider 2B assigned **Critical** priority with **62.17%** probability mass (`High: 34.84%`, `Medium: 2.20%`, `Low: 0.79%`), deriving a **49.56%** distribution concentration confidence score and a **0.7018 Noul score** requiring emergency out-of-band mitigation.
- **Policy Alignment**: The deterministic organizational risk policy (`v2026.1-policy`) evaluated Policy Rule 1 and validated that the model's Critical priority is fully aligned with enterprise compliance standards.

---

### Comprehensive Execution Provenance & Verification Matrix

The table below reports the actual, verified execution status of every component tested in this notebook:

| Component | Status | Evidence |
|---|---|---|
| **NVD retrieval** | `PASS (Live)` | HTTP 200 retrieval of CVSS 10.0 (Critical), vector `CVSS:3.1/AV:N/AC:L/PR:N/UI:N/S:C/C:H/I:H/A:H`, and 381 affected CPEs. |
| **CISA KEV retrieval** | `PASS (Live)` | Retrieved active in-the-wild exploitation listing under BOD 22-01 with authoritative mitigation directive. |
| **EPSS retrieval** | `PASS (Live)` | Retrieved empirical 100.00% 30-day in-wild exploitation probability estimate (top 100.00th percentile). |
| **Strands Decider** | `PASS (Live)` | Remote API HTTP 200 via `https://api.decider.cloud`: assigned `Critical` choice (62.17%), confidence 49.56%, emergency mitigation Noul 0.7018. |
| **Amazon Bedrock** | `FAIL (Live Probe)` | Probe failed with `LoginRefreshRequired` (active AWS profile session token expired; re-authentication required via `aws login`). |
| **End-to-end dual-model pipeline** | `DEGRADED (decider-only)` | Stage 1 succeeded, Stage 2 Bedrock failed; engine truthfully degraded to `decider-only` mode per production architecture. |
| **Synthetic parser tests** | `PASS (Fixture Test)` | Offline unit test verified `Stage2BedrockOutput` Pydantic schema validation, directive URL retention, and version `2.17.1` fact-checking disclosure. |
| **Risk-validation scenarios** | `PASS (Deterministic)` | Deterministic policy tests verified `ALIGNED`, `DISAGREEMENT` (mandates Critical for unlisted CVSS 9.8), and `INSUFFICIENT_EVIDENCE` (KEV 503). |

---

### Submission Details for AWS Builder Center

To submit this project to AWS Builder Center:
1. **GitHub Repository Path**:  
   `https://github.com/v-abhiramreddy/ThreatPrior-AI/blob/main/ThreatPrior_AI_Vulnerability_Prioritization.ipynb`
2. **Raw Notebook URL**:  
   `https://raw.githubusercontent.com/v-abhiramreddy/ThreatPrior-AI/main/ThreatPrior_AI_Vulnerability_Prioritization.ipynb`
3. **AWS Builder Center Instructions**:
   - Provide the GitHub repository URL or raw notebook URL in your project submission form.
   - The notebook contains pre-executed, verified cell outputs and can be run interactively in any JupyterLab, VS Code, or Amazon SageMaker Studio environment with Python 3.10+.